In [2]:
%load_ext dotenv
%dotenv

import sys
import os

# Get the project root (one level up from notebooks/)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [4]:
## Index data

from src.modules.search.aware_pts import SchemaAwareSemanticSearch


search_system = SchemaAwareSemanticSearch(backend_type="minsearch")
search_system.index_data(csv_path="../data/netflix_titles_enriched_full.csv")


In [19]:
from typing import List, Dict, Any

def full_asset_search(query: str, k=10) -> List[Dict[str, Any]]:
    results = search_system.search(query, top_k=k)

    docs = []
    for res in results:
        src = res.metadata
        doc = {
            "show_id": res.id,
            "type": res.content_type,
            "title": res.title,
            "director": src.get("director"),
            "cast": src.get("cast"),
            "country": src.get("country"),
            "date_added": src.get("date_added"),
            "release_year": src.get("release_year"),
            "rating": src.get("rating"),
            "duration": src.get("duration"),
            "listed_in": src.get("listed_in"),
            "description": src.get("description"),
        }
        docs.append(doc)
    return docs

In [15]:
my_query = "keanu reeves movies before 2005"
full_asset_search(my_query)

[{'show_id': 's8381',
  'type': 'Movie',
  'title': 'The Lake House',
  'director': 'Alejandro Agresti',
  'cast': 'Keanu Reeves, Sandra Bullock, Dylan Walsh, Shohreh Aghdashloo, Ebon Moss-Bachrach, Lynn Collins, Willeke van Ammelrooy, Christopher Plummer',
  'country': 'United States',
  'date_added': 'September 1, 2019',
  'release_year': 2006,
  'rating': 'PG',
  'duration': '98 min',
  'listed_in': 'Dramas, Romantic Movies, Sci-Fi & Fantasy',
  'description': "A lonely doctor begins writing letters to the frustrated architect who lives in her former home, only to discover that they're living two years apart."},
 {'show_id': 's8417',
  'type': 'Movie',
  'title': 'The Matrix Revolutions',
  'director': 'Lilly Wachowski, Lana Wachowski',
  'cast': 'Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss, Hugo Weaving, Jada Pinkett Smith, Mary Alice, Harold Perrineau, Monica Bellucci, Harry Lennix, Lambert Wilson, Nona Gaye',
  'country': 'United States',
  'date_added': 'November 1, 2019'

In [13]:
from openai import OpenAI
import openai
llm_client = OpenAI()


In [16]:
def generate_response(q):
    response = llm_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": q}]
    )

    json_response = response.choices[0].message.content
    return json_response

In [17]:
res = generate_response(my_query)
print(res)

Keanu Reeves has appeared in several notable films before 2005. Here’s a list of some of his key movies from that period:

1. **Bill & Ted's Excellent Adventure (1989)** - A sci-fi comedy about two slacker friends who travel through time.
2. **Bill & Ted's Bogus Journey (1991)** - The sequel to the original Bill & Ted film, featuring battles against evil robots.
3. **Point Break (1991)** - An action film about an FBI agent who goes undercover to catch a group of bank robbers.
4. **Bram Stoker's Dracula (1992)** - A horror film directed by Francis Ford Coppola, where Reeves plays Jonathan Harker.
5. **Much Ado About Nothing (1993)** - A film adaptation of Shakespeare's play, directed by Kenneth Branagh.
6. **Speed (1994)** - An action thriller where he plays a cop trying to stop a bomb on a bus.
7. **Johnny Mnemonic (1995)** - A cyberpunk film where he plays a data courier with a cybernetic brain implant.
8. **Chain Reaction (1996)** - A thriller about a scientist who becomes embroiled 

In [21]:
### RAG implementation

from string import Template

entry_template = Template("""
show_id: $show_id
type: $type
title: $title
director: $director
cast: $cast
country: $country
date_added: $date_added
release_year: $release_year
rating: $rating
duration: $duration
listed_in: $listed_in
description: $description
""")

prompt_template = Template("""
You are a streaming-catalog assistant.

Return ONE JSON object matching this SCHEMA exactly (no extra keys, no prose):

SCHEMA: {
  "catalog_recommendations": [
    "recommendation 1",
    "recommendation 2",
    "recommendation 3",
    "recommendation 4",
    "recommendation 5"
  ]
}

HARD RULES
- CONTEXT is pre-ranked (earlier = more relevant). Build "catalog_recommendations" ONLY from CONTEXT but feel free to change order.
- Scan CONTEXT top-down:
  1) Add items that plausibly match QUERY.
  2) If fewer than MIN_CATALOG and CONTEXT still has items, keep taking the next items (even weak matches)
     until you reach MIN_CATALOG or run out of CONTEXT.
- If CONTEXT has >= MIN_CATALOG items total, you MUST return at least MIN_CATALOG in "catalog_recommendations".
- Copy fields exactly from CONTEXT; for "cast", split the comma-separated string and trim; drop empties.
- Deduplicate by (title, release_year); keep the earlier one.
- Final counts:
  len(catalog_recommendations) >= min(MIN_CATALOG, number_of_items_in_CONTEXT)
- Recommendations must of the format "Title (Release Year): Small Description snippet"


INPUTS
QUERY: $query
MIN_CATALOG: $min_catalog

CONTEXT (ranked):
$context
""".strip())



def build_prompt(query, search_results, allow_external=True, min_catalog=5):
    context = ""
    for doc in search_results:
        context += entry_template.substitute(**doc) + "\n\n"
    return prompt_template.substitute(
        query=query,
        context=context,
        allow_external=str(allow_external).lower(), 
        min_catalog=min_catalog,
    )

def llm(prompt):
    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

def rag(query):
    search_results = full_asset_search(query, k=10)
    if not search_results:
        print("WARN: No relevant results found.")

    prompt = build_prompt(query, search_results)
    response = llm(prompt)
    return response


In [82]:
fr = rag("Movies directed by Christopher Nolan")

In [83]:
import json
print (json.dumps(json.loads(fr), indent=2))

{
  "catalog_recommendations": [
    "Even the Rain (2010): While making a film about the incursion of Christopher Columbus in the New World, a director finds the Bolivian locals protesting modern exploitation.",
    "Colkatay Columbus (2016): When Christopher Columbus mysteriously appears in modern-day Kolkata, India, two struggling young men look to him for advice on achieving success.",
    "Captain Underpants Epic Choice-o-Rama (2020): In this interactive special, Harold and George need your decision-making skills to stop Krupp from blowing their beloved treehouse to smithereens.",
    "Teenage Mutant Ninja Turtles (2007): In this animated adventure, Master Splinter whips the four Ninja Turtles back into shape to defeat monsters running amok in New York.",
    "House of the Witch (2017): A group of daring teens finds themselves in a fight for their lives inside a haunted house when a sinister spirit crashes their Halloween party."
  ]
}


### LLM as a judge

In [27]:
prompt_judge_template = """
You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system.
Your task is to analyze the relevance of the generated content to the user's question.
Based on the relevance of the available content, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [50]:
import json

with open("../data/ground_truth/new_ground_truth.json", "r") as f:
    json_data = json.load(f)

gt = []
for item, values in json_data.items():
    for val in values:
        gt.append({'id': item, 'question': val})


gt[:10]

[{'id': 's1426', 'question': 'documentaries about chess'},
 {'id': 's1426', 'question': 'behind the scenes of popular shows'},
 {'id': 's1426', 'question': 'short movies from 2021'},
 {'id': 's1426', 'question': 'TV 14 rated documentaries'},
 {'id': 's1426', 'question': 'recent US documentaries'},
 {'id': 's7321', 'question': 'independent comedies about family'},
 {'id': 's7321', 'question': 'feel good movies about faith'},
 {'id': 's7321', 'question': 'films featuring Ally Sheedy'},
 {'id': 's7321', 'question': 'stories about returning home'},
 {'id': 's7321', 'question': 'movies from 2016'}]

In [51]:
ground_truth = gt

In [56]:
from tqdm.auto import tqdm
from random import sample

In [70]:
evaluations = {}
evals = []

In [71]:
for record in tqdm(sample(ground_truth, k=100)):

    question = record['question']
    answer_llm = rag(question)


    prompt_judge = prompt_judge_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt_judge)
    print(evaluation)

    evals.append((record, answer_llm, evaluation))

  0%|          | 0/100 [00:00<?, ?it/s]

{
  "Relevance": "RELEVANT",
  "Explanation": "All films listed in the generated answer feature George Clooney, directly addressing the user's question about films with him. Each title accurately represents his work, making the response fully relevant." 
}
{
  "Relevance": "RELEVANT",
  "Explanation": "The generated answer lists independent films that directly relate to the theme of family secrets, as requested. Each recommendation addresses aspects of familial relationships and hidden truths, aligning well with the user's question."
}
{
  "Relevance": "RELEVANT",
  "Explanation": "The generated answer provides a list of documentaries that are specifically about racing, which directly addresses the user's request for sports documentaries related to that theme. Each recommendation pertains to different aspects of racing, whether it's the sport itself, specific drivers, or historical events in racing, all of which fulfill the user's query."
}
{
  "Relevance": "RELEVANT",
  "Explanation":

In [73]:
evals[:5]

[({'id': 's6879', 'question': 'films with George Clooney'},
  '{\n  "catalog_recommendations": [\n    "The Midnight Sky (2020): In the aftermath of a global catastrophe, a lone scientist in the Arctic races to contact a crew of astronauts with a warning not to return to Earth.",\n    "The American (2010): Dispatched to a small Italian town to await further orders, assassin Jack embarks on a double life that may be more relaxing than is good for him.",\n    "The Peacemaker (1997): After terrorists trigger a nuclear blast in Russia, a U.S. Special Forces intelligence agent and a nuclear weapons expert come to the rescue.",\n    "Up in the Air (2009): Ryan Bingham flies around the country firing employees on behalf of companies, but he faces losing the job he savors to recent college grad Natalie.",\n    "Good Night, and Good Luck (2005): Veteran television newsman Edward R. Murrow faces off against Sen. Joseph McCarthy and his crusade to quell the threat of communism in America."\n  ]\n}

In [74]:
import pandas as pd
edf = pd.DataFrame(evals, columns=['record', 'answer_llm', 'evaluation'])
edf.head()

,record,answer_llm,evaluation
0,"{'id': 's6879', 'question': 'films with George...","{\n ""catalog_recommendations"": [\n ""The Mi...","{\n ""Relevance"": ""RELEVANT"",\n ""Explanation""..."
1,"{'id': 's4125', 'question': 'independent films...","{\n ""catalog_recommendations"": [\n ""The Tr...","{\n ""Relevance"": ""RELEVANT"",\n ""Explanation""..."
2,"{'id': 's2257', 'question': 'sports documentar...","{\n ""catalog_recommendations"": [\n ""Speed ...","{\n ""Relevance"": ""RELEVANT"",\n ""Explanation""..."
3,"{'id': 's2307', 'question': 'series about civi...","{\n ""catalog_recommendations"": [\n ""Hell o...","{\n ""Relevance"": ""RELEVANT"",\n ""Explanation""..."
4,"{'id': 's5405', 'question': 'shows featuring C...","{\n ""catalog_recommendations"": [\n ""The Tr...","{\n ""Relevance"": ""NON_RELEVANT"",\n ""Explanat..."


In [75]:
edf['id'] = edf['record'].apply(lambda x: x['id'])
edf['question'] = edf['record'].apply(lambda x: x['question'])
edf['relevance'] = edf['evaluation'].apply(lambda x: json.loads(x).get('Relevance'))
edf['explanation'] = edf['evaluation'].apply(lambda x: json.loads(x).get('Explanation'))
edf = edf.drop(columns=['record','evaluation'])


In [76]:
edf.head()

,answer_llm,id,question,relevance,explanation
0,"{\n ""catalog_recommendations"": [\n ""The Mi...",s6879,films with George Clooney,RELEVANT,All films listed in the generated answer featu...
1,"{\n ""catalog_recommendations"": [\n ""The Tr...",s4125,independent films about family secrets,RELEVANT,The generated answer lists independent films t...
2,"{\n ""catalog_recommendations"": [\n ""Speed ...",s2257,sports documentaries about racing,RELEVANT,The generated answer provides a list of docume...
3,"{\n ""catalog_recommendations"": [\n ""Hell o...",s2307,series about civil war struggles,RELEVANT,The generated answer provides a list of series...
4,"{\n ""catalog_recommendations"": [\n ""The Tr...",s5405,shows featuring Chen Hanwei,NON_RELEVANT,The generated answer lists two shows that do n...


In [78]:
edf.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.48
PARTLY_RELEVANT    0.39
NON_RELEVANT       0.13
Name: proportion, dtype: float64

In [80]:
edf[edf.relevance == 'NON_RELEVANT']

,answer_llm,id,question,relevance,explanation
4,"{\n ""catalog_recommendations"": [\n ""The Tr...",s5405,shows featuring Chen Hanwei,NON_RELEVANT,The generated answer lists two shows that do n...
8,"{\n ""catalog_recommendations"": [\n ""GANTZ:...",s6849,intense japanese features,NON_RELEVANT,The generated answer lists various anime films...
16,"{\n ""catalog_recommendations"": [\n ""Monkey...",s4663,international titles starring Phakin Khamwilaisak,NON_RELEVANT,The generated answer does not mention Phakin K...
18,"{\n ""catalog_recommendations"": [\n ""Lock (...",s4582,films featuring Gippy Grewal,NON_RELEVANT,"The generated answer lists several films, but ..."
23,"{\n ""catalog_recommendations"": [\n ""Udaan ...",s6521,series featuring Ram Kapoor,NON_RELEVANT,The generated answer does not mention Ram Kapo...
44,"{\n ""catalog_recommendations"": [\n ""Bedtim...",s7303,stories about shapeshifters,NON_RELEVANT,The generated answer does not contain any cont...
61,"{\n ""catalog_recommendations"": [\n ""All of...",s4199,romantic dramas with Jennylyn Mercado,NON_RELEVANT,The generated answer lists romantic dramas tha...
74,"{\n ""catalog_recommendations"": [\n ""Nura: ...",s7628,jun fukuyama voice acting,NON_RELEVANT,The generated content does not address the use...
81,"{\n ""catalog_recommendations"": [\n ""The Ki...",s5178,films starring Diogo Morgado,NON_RELEVANT,The generated answer does not mention any film...
88,"{\n ""catalog_recommendations"": [\n ""My Bir...",s2738,movies by Rajiv Chilaka,NON_RELEVANT,The generated answer provides a list of movies...


In [81]:
# >>>  Pretty decent for which we didn't get good results


In [86]:
gdf = pd.DataFrame(ground_truth)
gdf.head()

,id,question
0,s1426,documentaries about chess
1,s1426,behind the scenes of popular shows
2,s1426,short movies from 2021
3,s1426,TV 14 rated documentaries
4,s1426,recent US documentaries


In [95]:
gdf.to_csv("../data/ground_truth_retrieval.csv", index=False)

